# Updated Gemini-3-Flash-Preview ChildSnack (LLM planner, single-sandwich-line direct unraveling)

In [ ]:
# Cell 1: Setup (Gemini planner pipeline, ChildSnack single-sandwich-line independent)
import os
import re
import json
import time
import subprocess
import platform
from pathlib import Path
from datetime import datetime

from dotenv import load_dotenv
from google import genai
from google.genai import types

# Project root (run notebook from formalizer-scaling/src/planner_gen_notebooks/ChildSnack/single_sandwich_line)
PROJECT_ROOT = Path.cwd().parents[3]

# Data / prompt / output paths
DATA_DIR = PROJECT_ROOT / "data"
PDDL_DIR = DATA_DIR / "pddl_unraveling_domain" / "ChildSnack" / "single_sandwich_line"
DESC_DIR = DATA_DIR / "descriptions_unraveling_domain" / "ChildSnack" / "single_sandwich_line"

DOMAIN_PROMPT_PATH = PROJECT_ROOT / "prompts" / "ChildSnack" / "llm_planner_instruction.txt"
FALLBACK_PROMPT_PATH = PROJECT_ROOT / "prompts" / "BlocksWorld" / "llm_planner_instruction.txt"
PROMPT_PATH = DOMAIN_PROMPT_PATH if DOMAIN_PROMPT_PATH.exists() else FALLBACK_PROMPT_PATH

MODEL_DIRNAME = "updated_gemini_3_flash"
OUTPUT_ROOT = PROJECT_ROOT / "output" / "llm_planner_direct_unraveling" / "ChildSnack" / "single_sandwich_line" / MODEL_DIRNAME
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Project-local VAL executable for the current operating system
VAL_BUILD_PLATFORM = "macos64" if platform.system() == "Darwin" else "linux64"
VAL_PATH = PROJECT_ROOT / "VAL" / "build" / VAL_BUILD_PLATFORM / "Release" / "bin" / "Validate"

# Model + API
MODEL_NAME = "gemini-3-flash-preview"
ENV_PATH = PROJECT_ROOT / ".env"
load_dotenv(ENV_PATH)

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
    raise ValueError(f"GOOGLE_API_KEY not found in {ENV_PATH}")

client = genai.Client(api_key=GOOGLE_API_KEY)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("VAL exists:", VAL_PATH.exists())
print("PDDL dir exists:", PDDL_DIR.exists())
print("Description dir exists:", DESC_DIR.exists())
print("Prompt exists:", PROMPT_PATH.exists())
print("Model:", MODEL_NAME)

In [ ]:
# Cell 2: Load problem IDs and planner instruction
import pandas as pd

problem_files = sorted(PDDL_DIR.glob("p*_problem.pddl"))
PROBLEM_IDS = [p.name.removesuffix("_problem.pddl") for p in problem_files]

PLANNER_INSTRUCTION = PROMPT_PATH.read_text()

print("Num problems:", len(PROBLEM_IDS))
if PROBLEM_IDS:
    print("First/last:", PROBLEM_IDS[0], PROBLEM_IDS[-1])
print("Planner prompt chars:", len(PLANNER_INSTRUCTION))
print("PDDL dir exists:", PDDL_DIR.exists())
print("Description dir exists:", DESC_DIR.exists())

In [ ]:
# Cell 3: Helper - load NL descriptions and ground-truth PDDL for one problem ID
def load_problem_artifacts(problem_id: str):
    domain_desc_path = DESC_DIR / f"{problem_id}_domain.txt"
    problem_number = int(problem_id.removeprefix("p"))
    description_suffix = "clean" if problem_number <= 20 else "clean2"
    problem_desc_path = DESC_DIR / f"{problem_id}_problem_{description_suffix}.txt"
    gt_domain_pddl_path = PDDL_DIR / "domain.pddl"
    gt_problem_pddl_path = PDDL_DIR / f"{problem_id}_problem.pddl"

    if not domain_desc_path.exists():
        raise FileNotFoundError(f"Missing domain description: {domain_desc_path}")
    if not problem_desc_path.exists():
        raise FileNotFoundError(f"Missing problem description: {problem_desc_path}")
    if not gt_domain_pddl_path.exists():
        raise FileNotFoundError(f"Missing ground-truth domain PDDL: {gt_domain_pddl_path}")
    if not gt_problem_pddl_path.exists():
        raise FileNotFoundError(f"Missing ground-truth problem PDDL: {gt_problem_pddl_path}")

    return {
        "domain_desc": domain_desc_path.read_text(),
        "problem_desc": problem_desc_path.read_text(),
        "gt_domain_pddl_path": gt_domain_pddl_path,
        "gt_problem_pddl_path": gt_problem_pddl_path,
        "gt_domain_pddl_text": gt_domain_pddl_path.read_text(),
    }

In [ ]:
# Cell 4: Helper - build planner prompt (NL domain + NL problem + GT domain PDDL)
def build_prompt(problem_id: str):
    artifacts = load_problem_artifacts(problem_id)

    return (
        PLANNER_INSTRUCTION
        + "\n\nYou are an expert automated planner.\n"
        "Create an executable plan that is consistent with all provided information.\n\n"
        "### Natural-language domain description\n"
        f"{artifacts['domain_desc']}\n\n"
        "### Natural-language problem description\n"
        f"{artifacts['problem_desc']}\n\n"
        "### Ground-truth domain PDDL\n"
        f"{artifacts['gt_domain_pddl_text']}\n\n"
        "Output only the final plan wrapped in <plan>...</plan>."
    )

In [ ]:
# Cell 5: Helper - call Gemini for direct plan generation
def call_gemini(prompt: str):
  resp = client.models.generate_content(
      model=MODEL_NAME,
      contents=prompt,
      config=types.GenerateContentConfig(
          temperature=0.4,
          max_output_tokens=200000,
          thinking_config={
              "thinking_level": "medium"
          },
      ),
  )
  return resp.text or ""

In [ ]:
# Cell 6: Helper - extract <plan>...</plan> block
def extract_plan(raw_text: str):
  if not raw_text:
      return ""

  m = re.search(r"<plan>(.*?)</plan>", raw_text, flags=re.DOTALL | re.IGNORECASE)
  if not m:
      return ""

  plan_text = m.group(1).strip()

  # Keep one action per non-empty line
  lines = [ln.strip() for ln in plan_text.splitlines() if ln.strip()]
  return "\n".join(lines)

In [ ]:
# Cell 7: Helper - validate a plan with VAL (GT domain + GT problem)
def run_val(domain_pddl_path: Path, problem_pddl_path: Path, plan_path: Path):
  if not VAL_PATH.exists():
      return {
          "val_ok": False,
          "val_output": f"VAL executable not found: {VAL_PATH}",
      }

  cmd = [str(VAL_PATH), "-v", str(domain_pddl_path), str(problem_pddl_path), str(plan_path)]
  proc = subprocess.run(cmd, capture_output=True, text=True)

  out_text = (proc.stdout or "").strip()
  err_text = (proc.stderr or "").strip()
  merged = (out_text + ("\n" + err_text if err_text else "")).strip()

  val_ok = "Plan valid" in merged
  return {
      "val_ok": val_ok,
      "val_output": merged,
  }

In [ ]:
# Cell 8: Helper - run one problem end-to-end (generate plan + VAL), persist outputs
def process_one_problem(problem_id: str):
    artifacts = load_problem_artifacts(problem_id)
    prompt = build_prompt(problem_id)
    raw_text = call_gemini(prompt)
    plan_text = extract_plan(raw_text)

    cand_dir = OUTPUT_ROOT / problem_id / "cand_00"
    cand_dir.mkdir(parents=True, exist_ok=True)

    raw_path = cand_dir / "raw.txt"
    plan_path = cand_dir / "plan.txt"
    val_log_path = cand_dir / "validator.log"
    err_path = cand_dir / "planner_error.txt"

    raw_path.write_text(raw_text or "")
    (cand_dir / "domain.pddl").write_text(artifacts["gt_domain_pddl_text"])
    (cand_dir / "problem.pddl").write_text(artifacts["gt_problem_pddl_path"].read_text())

    if not plan_text:
        plan_path.write_text("")
        err_msg = "Empty or missing <plan>...</plan> output."
        err_path.write_text(err_msg)
        return {
            "problem": problem_id,
            "plan_found": False,
            "val_ok": False,
            "planner_error": err_msg,
        }

    plan_path.write_text(plan_text + ("" if plan_text.endswith("\n") else "\n"))

    val_result = run_val(artifacts["gt_domain_pddl_path"], artifacts["gt_problem_pddl_path"], plan_path)
    val_log_path.write_text(val_result.get("val_output", "") or "")

    return {
        "problem": problem_id,
        "plan_found": True,
        "val_ok": bool(val_result["val_ok"]),
        "planner_error": "",
    }


In [ ]:
# Cell 9: Generate only (no VAL)
smoke_problem = "p01"
artifacts = load_problem_artifacts(smoke_problem)

prompt = build_prompt(smoke_problem)
raw_text = call_gemini(prompt)
plan_text = extract_plan(raw_text)

cand_dir = OUTPUT_ROOT / smoke_problem / "cand_00"
cand_dir.mkdir(parents=True, exist_ok=True)

(cand_dir / "raw.txt").write_text(raw_text or "")
(cand_dir / "plan.txt").write_text((plan_text + "\n") if plan_text else "")
(cand_dir / "domain.pddl").write_text(artifacts["gt_domain_pddl_text"])
(cand_dir / "problem.pddl").write_text(artifacts["gt_problem_pddl_path"].read_text())

print("Generation-only done.")
print("problem:", smoke_problem)
print("extracted_plan_chars:", len(plan_text or ""))
print("saved:", cand_dir)

In [ ]:
# Cell 10: VAL only (no model call), concise output
import contextlib
import io

smoke_problem = "p01"
artifacts = load_problem_artifacts(smoke_problem)
cand_dir = OUTPUT_ROOT / smoke_problem / "cand_00"

plan_path = cand_dir / "plan.txt"
if not plan_path.exists():
    raise FileNotFoundError(f"Missing plan file: {plan_path}")

plan_text = plan_path.read_text().strip()
if not plan_text:
    summary = {
        "problem": smoke_problem,
        "plan_found": False,
        "val_ok": False,
        "planner_error": "Empty plan.txt (no actions).",
    }
    (cand_dir / "planner_error.txt").write_text(summary["planner_error"])
    print("Produced files:")
    print("- planner_error.txt")
    print("VAL result: not run (empty plan)")
    print("Summary:")
    print(json.dumps(summary, indent=2))
else:
    with contextlib.redirect_stdout(io.StringIO()):
        val_result = run_val(artifacts["gt_domain_pddl_path"], artifacts["gt_problem_pddl_path"], plan_path)

    (cand_dir / "validator.log").write_text(val_result.get("val_output", "") or "")

    summary = {
        "problem": smoke_problem,
        "plan_found": True,
        "val_ok": bool(val_result["val_ok"]),
        "planner_error": "",
    }

    print("Produced files:")
    print("- validator.log")
    print(f"VAL result: {'valid' if val_result.get('val_ok') else 'invalid'}")
    print("Summary:")
    print(json.dumps(summary, indent=2))

In [ ]:
# Cell 11: Batch generate plans only (no VAL)
from tqdm.auto import tqdm

BATCH_PROBLEMS_GEN = [f"p{i:02d}" if i < 100 else f"p{i}" for i in range(2, 81)]

gen_rows = []

for pid in tqdm(BATCH_PROBLEMS_GEN, desc="Generating plans"):
    artifacts = load_problem_artifacts(pid)
    prompt = build_prompt(pid)
    raw_text = call_gemini(prompt)
    plan_text = extract_plan(raw_text)

    cand_dir = OUTPUT_ROOT / pid / "cand_00"
    cand_dir.mkdir(parents=True, exist_ok=True)

    (cand_dir / "raw.txt").write_text(raw_text or "")
    (cand_dir / "plan.txt").write_text((plan_text + "\n") if plan_text else "")
    (cand_dir / "domain.pddl").write_text(artifacts["gt_domain_pddl_text"])
    (cand_dir / "problem.pddl").write_text(artifacts["gt_problem_pddl_path"].read_text())

    gen_rows.append(
        {
            "problem": pid,
            "plan_extracted": bool(plan_text),
            "plan_chars": len(plan_text or ""),
            "cand_dir": str(cand_dir),
        }
    )

print("Batch generation done.")
print("Problems:", BATCH_PROBLEMS_GEN[0], "->", BATCH_PROBLEMS_GEN[-1], f"({len(BATCH_PROBLEMS_GEN)})")
print("Plans extracted:", sum(1 for r in gen_rows if r["plan_extracted"]), "/", len(gen_rows))

In [ ]:
# Cell 12: Batch VAL only (no model call)
from tqdm.auto import tqdm
import contextlib
import io

BATCH_PROBLEMS_VAL = [f"p{i:02d}" if i < 100 else f"p{i}" for i in range(71, 81)]

val_rows = []

for pid in tqdm(BATCH_PROBLEMS_VAL, desc="Running VAL"):
    artifacts = load_problem_artifacts(pid)
    cand_dir = OUTPUT_ROOT / pid / "cand_00"
    plan_path = cand_dir / "plan.txt"

    row = {
        "problem": pid,
        "plan_found": False,
        "val_ok": False,
        "planner_error": "",
    }

    if not plan_path.exists():
        row["planner_error"] = f"Missing plan file: {plan_path}"
        (cand_dir / "planner_error.txt").write_text(row["planner_error"])
        val_rows.append(row)
        continue

    plan_text = plan_path.read_text().strip()
    if not plan_text:
        row["planner_error"] = "Empty plan.txt (no actions)."
        (cand_dir / "planner_error.txt").write_text(row["planner_error"])
        val_rows.append(row)
        continue

    row["plan_found"] = True

    with contextlib.redirect_stdout(io.StringIO()):
        val_result = run_val(artifacts["gt_domain_pddl_path"], artifacts["gt_problem_pddl_path"], plan_path)

    (cand_dir / "validator.log").write_text(val_result.get("val_output", "") or "")
    row["val_ok"] = bool(val_result["val_ok"])

    val_rows.append(row)

print("Batch VAL done.")
print("Problems:", BATCH_PROBLEMS_VAL[0], "->", BATCH_PROBLEMS_VAL[-1], f"({len(BATCH_PROBLEMS_VAL)})")
print("Plan found:", sum(int(r["plan_found"]) for r in val_rows), "/", len(val_rows))
print("VAL valid:", sum(int(r["val_ok"]) for r in val_rows), "/", len(val_rows))